# Reti neurali profonde



## Preparazione in Colab

1. Salva una copia del notebook nel tuo Drive.
2. Esegui la cella seguente e attendi «Preparazione completata».
3. Prosegui dall’alto verso il basso e completa gli esercizi.

La cella carica i dataset e i controlli dal repository del corso. Non devi installare Python né caricare manualmente i CSV. Se la sessione viene riavviata, esegui nuovamente la preparazione.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
from urllib.request import urlopen
from IPython.display import display, Markdown

course_source = 'https://raw.githubusercontent.com/prof-davidaffo/kaggle-learntools/master'
course_root = (Path("/content") if Path("/content").is_dir() else Path.home()) / "ai-lab-colab"
course_root.mkdir(parents=True, exist_ok=True)
bootstrap_path = course_root / "_bootstrap.py"
bootstrap_hash = '56f1300be5bf273a70dd4a30e704b20f642ec01dacc4d2d4b5a295a02e1da60f'
if not bootstrap_path.is_file() or hashlib.sha256(bootstrap_path.read_bytes()).hexdigest() != bootstrap_hash:
    with urlopen(course_source + "/ai_lab/tools/colab_runtime.py", timeout=90) as response:
        bootstrap_data = response.read()
    if hashlib.sha256(bootstrap_data).hexdigest() != bootstrap_hash:
        raise RuntimeError("Course setup changed. Reopen the latest notebook from the course site.")
    bootstrap_path.write_bytes(bootstrap_data)
bootstrap_spec = importlib.util.spec_from_file_location("ai_lab_colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(bootstrap_spec)
bootstrap_spec.loader.exec_module(bootstrap)
bootstrap.download_runtime(course_source + "/ai_lab/colab/runtime.json", course_source, course_root)
bootstrap.setup(course_root, globals(), deep_learning=True)
display(Markdown("**Preparazione completata.** Puoi eseguire le celle della lezione."))


# Introduzione
Come si costruisce una rete capace di apprendere relazioni complesse? La chiave è la **modularità**: combinare unità semplici per ottenere una trasformazione complessa.

# Layer
I neuroni sono organizzati in **strati**. Un gruppo di unità lineari che riceve gli stessi ingressi forma un layer **Dense**, completamente connesso.

![Due unità dense collegate agli ingressi e al bias](https://storage.googleapis.com/kaggle-media/learn/images/2MA4iMV.png)

Ogni strato esegue una trasformazione relativamente semplice. Una sequenza profonda combina le trasformazioni, avvicinando progressivamente gli ingressi all'uscita desiderata.

In Keras un layer può rappresentare molte operazioni: connessioni [convoluzionali](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D) o [ricorrenti](https://www.tensorflow.org/api_docs/python/tf/keras/layers/RNN), trasformazioni delle feature come [Embedding](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding), oppure [semplici somme](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Add). Esplora il [catalogo dei layer](https://www.tensorflow.org/api_docs/python/tf/keras/layers).

# Funzioni di attivazione
Due layer lineari consecutivi restano equivalenti a una trasformazione lineare: non bastano a rappresentare curve. Serve una **non linearità**, introdotta dalla funzione di attivazione applicata alle uscite del layer.

![Relazioni lineari e non lineari](https://storage.googleapis.com/kaggle-media/learn/images/OLSUEYT.png)

Una funzione molto comune è il rettificatore **\(\max(0,x)\)**: lascia invariati i valori positivi e porta a zero quelli negativi.

![Grafico del rettificatore](https://storage.googleapis.com/kaggle-media/learn/images/aeIyAlF.png)

Applicato a un'unità lineare produce una **ReLU**, *Rectified Linear Unit*: l'uscita è **`max(0, w * x + b)`**. La «piega» del grafico permette di uscire dal mondo delle sole rette.

![Unità ReLU](https://storage.googleapis.com/kaggle-media/learn/images/eFry7Yu.png)


# Sovrapporre layer Dense
Con le attivazioni non lineari possiamo costruire trasformazioni complesse.

![Ingressi, layer nascosti e uscita](https://storage.googleapis.com/kaggle-media/learn/images/Y5iwFQZ.png)

Gli strati prima dell'uscita si chiamano **hidden layer**, o strati nascosti, perché le loro uscite non sono direttamente il risultato finale.

Qui l'ultimo layer è lineare, senza attivazione: è adatto alla regressione, nella quale prevediamo un valore numerico. La classificazione può richiedere un'attivazione diversa in uscita.

## Modelli Sequential
`Sequential` collega i layer nell'ordine della lista: il primo riceve i dati e l'ultimo produce l'uscita. Il codice seguente costruisce la rete della figura.


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    # Layer nascosti con attivazione ReLU.
    layers.Dense(units=4, activation='relu', input_shape=[2]),
    layers.Dense(units=3, activation='relu'),
    # Layer di output lineare.
    layers.Dense(units=1),
])

Passa i layer come un'unica lista, **`[layer, layer, ...]`**, non come argomenti separati. Specifica il nome dell'attivazione nell'argomento **`activation`**.

Ora **[costruisci una rete profonda](https://prof-davidaffo.github.io/kaggle-learntools/corsi/deep_learning_intro/ex2.html)** per il dataset *Concrete*.
